In [1]:
#1 Installations

!pip install requests beautifulsoup4 pandas openpyxl

In [1]:
#2 Imports

import requests
import time
import random
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, quote

import xml.etree.ElementTree as ET

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
#3  Headers

DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [14]:
#4 List of search terms for URL-completion, will later be encoded in the main loop

keywords = [
    "CCS",
    "CCU",
    "CCUS",
    "CCU/S",
    "Carbon Management",
    "Carbon Capture",
    "Carbon Capture Utilisation and Storage",
    "Carbon Capture Utilization and Storage",
    "Carbon Capture and Utilisation",
    "Carbon Capture and Utilization",
    "Abscheidung",
    "Kohlenstoffabscheidung",
    "Technische Senke",
    "KSpG",
    "KSpTG",
    "Kohlendioxid-Speicherungsgesetz",
    "Kohlendioxid-Speicherung-und-Transport-Gesetz"
]


In [4]:
#5  Categorisation of URL types
# A: homepage-searches without pagination
# B: homepage-searches with pagination
# C: sitemap fallback domains

def get_url_type(url, source_id=None):

    if source_id is not None and source_id in SITEMAP_FALLBACK_DOMAINS:
        return "C"

    if "{PageP}" in url:
        return "B"

    return "A"

In [5]:
#6  List of search URLS

urls = {
1:"https://www.cemex.de/search?q={KeywordP}&delta=99",
2:"https://bdi.eu/de/suche?page={PageP}&search={KeywordP}", 
3:"https://www.harbourenergy.com/search/?query={KeywordP}&submit=&page={PageP}",
4:"https://green-planet-energy.de/suche?query={KeywordP}&tx_kesearch_pi1%5Bpage%5D={PageP}",
5:"https://www.vci.de",
6:"https://fortschrittinfreiheit.de/page/{PageP}/?s={KeywordP}",
7:"https://suche.agora-industrie.de/suche?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
8:"https://power-shift.de/page/{PageP}/?s={KeywordP}",
9:"https://www.dnr.de/suche?search={KeywordP}&page={PageP}",
10:"https://www.wwf.de/suche?s%5Bpage%5D={PageP}&s%5Bq%5D={KeywordP}",
11:"https://www.nabu.de/modules/suche/htdig.php?htdig%5Bwords%5D={KeywordP}&page={PageP}",
12:"https://www.umweltrat.de/SiteGlobals/Forms/Suche/DE/Servicesuche/Servicesuche_Formular.html?templateQueryString={KeywordP}&resultsPerPage=99",
13:"https://de.bellona.org/page/{PageP}/?s={KeywordP}",
14:"https://www.staedtetag.de/suchergebnisse?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
15:"https://www.ruhr-uni-bochum.de",
16:"https://www.landkreistag.de/suche?q={KeywordP}&start={PageP}0",
17:"https://www.dstgb.de/?q={KeywordP}",
18:"https://www.bund.net/service/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
19:"https://www.lyondellbasell.com/en/search/?keyword={KeywordP}",
20:"https://www.basf.com",
21:"https://www.bde.de/search/?q={KeywordP}",
22:"https://www.eutop.com",
23:"https://www.bund-sachsen.de/service/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
24:"https://www.bund-nrw.de/suchergebnis/?tx_solr%5Bfilter%5D%5B0%5D=&tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}&tx_solr%5Bsort%5D=relevance+desc",
25:"https://www.baustoffindustrie.de/search?query={KeywordP}",
26:"https://www.kalk.de",
27:"https://www.bveg.de/page/{PageP}/?s={KeywordP}&facet_type&sort_type=relevance#038;facet_type&sort_type=relevance",
28:"https://www.campact.de/page/{PageP}/?s={KeywordP}",
29:"https://www.cm-allianz.de",
30:"https://carbon-management-initiative.de",
31:"https://www.duh.de/suche/?no_cache=1&tx_kesearch_pi1%5Bsword%5D={KeywordP}&tx_kesearch_pi1%5Bpage%5D={PageP}",
32:"https://dvne.org",
33:"https://www.verkehrsforum.de/de/suchergebnisse?psearch={KeywordP}",
34:"https://gas-h2.de",
35:"https://www.papierindustrie.de",
36:"https://www.dvgw.de",
37:"https://www.dr-martin-eckert.de",
38:"https://www.thyssenkrupp-uhde.com",
39:"https://www.thyssenkrupp-polysius.com",
40:"https://www.eew-energyfromwaste.com/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
41:"https://www.equinor.de",
42:"https://corporate.exxonmobil.com/search?search={KeywordP}",
43:"https://www.everllence.com/search-results?search={KeywordP}&search_page={PageP}",
44:"https://www.evonik.com/de/search.html?q={KeywordP}&languages=German%3BEnglish",
45:"https://www.faktor3.de",
46:"https://www.fluxys.com/de/search#search_e={PageP}0&search_q={KeywordP}",
47:"https://forumue.de/page/{PageP}/?s={KeywordP}",
48:"https://www.handundfusser.de",
49:"https://www.germanwatch.org/de/suche?text={KeywordP}",
50:"https://germanzero.de",
51:"https://www.h2ercules.com",
52:"https://www.portofrotterdam.com",
53:"https://www.helmholtz-klima.de/#o=search&q={KeywordP}",
54:"https://www.holcim.de/search-results?search_api_fulltext={KeywordP}&page={PageP}",
55:"https://www.ihk-nord.de",
56:"https://www.itad.de/@@search?sort_on=relevance&sort_order=&SearchableText={KeywordP}&advanced_search=False&pt_toggle=%23&portal_type:list=File&portal_type:list=Newsletter&portal_type:list=Event&portal_type:list=Collection&portal_type:list=Folder&portal_type:list=Image&portal_type:list=Newsletter%20Issue&portal_type:list=Document&portal_type:list=Fragebogen&portal_type:list=Link&portal_type:list=Anlage&_=1773481589563&b_start:int={PageP}0",
57:"https://www.leag.de/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
58:"https://www.lhoist.com/de-DE/suche?2026prod_search%5Bquery%5D={KeywordP}&2026prod_search%5Bpage%5D={PageP}",
59:"https://www.linde-gas.de",
60:"https://www.mew-verband.de/suche?query={KeywordP}&ccm_paging_p={PageP}&ccm_order_by=&ccm_order_by_direction=",
61:"https://www.gasunie.nl/en/search-the-website?q={KeywordP}&ss360PageLink=1&ss360Offset={PageP}0",
62:"https://www.ontras.com/de/search/node?keys={KeywordP}&page={PageP}",
63:"https://oge.net/de/suche-seite{PageP}?search={KeywordP}",
64:"https://www.rostock-port.de",
65:"https://www.rwe.com/suche/?q={KeywordP}",
66:"https://www.shell.de/search.html?q={KeywordP}&offset={PageP}0",
67:"https://www.swm.de/suche?suchbegriff={KeywordP}",
68:"https://www.thepartners.io",
69:"https://tes-h2.com",
70:"https://www.tuev-nord.de/de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
71:"https://www.tuv.com/germany/de/search.html#?term={KeywordP}&page_offset={PageP}",
72:"https://www.tuev-verband.de/suche?tuev%5Bpage%5D={PageP}&tuev%5Bq%5D={KeywordP}",
73:"https://www.uniper.energy/de/search?query={KeywordP}&page={PageP}",
74:"https://www.urgewald.org/search/content?keys={KeywordP}&page={PageP}",
75:"https://www.vais.de",
76:"https://www.vdma.eu/de/suche#search-page-tabbar-articles-id?searchKeyword={KeywordP}&sorts=RELEVANCE",
77:"https://www.vgms.de/suche?tx_kesearch_pi1%5Bpage%5D={PageP}&tx_kesearch_pi1%5BsortByDir%5D=asc&tx_kesearch_pi1%5Bsword%5D={KeywordP}",
78:"https://www.vdz-online.de",
79:"https://www.vbw-bayern.de/vbw/Suche.jsp.is?queryText={KeywordP}&co=3&pageSize=99&sort=Score",
80:"https://vik.de/search?search={KeywordP}",
81:"https://www.vku.de/suche/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
82:"https://www.vbcoll.de",
83:"https://www.efuel-alliance.eu/search?tx_kesearch_pi1%5Bfilter_3_end%5D=&tx_kesearch_pi1%5Bfilter_3_start%5D=&tx_kesearch_pi1%5Bpage%5D={PageP}&tx_kesearch_pi1%5Bsword%5D={KeywordP}",
84:"https://www.bernd-westphal.de/page/{PageP}/?s={KeywordP}",
85:"https://wirtschaftsrat.de",
86:"https://en2x.de/page/{PageP}/?s={KeywordP}",
87:"https://zds-seehaefen.de",
88:"https://www.bayernets.de",
89:"https://www.zdh.de/suchergebnisse/?tx_solr%5Bpage%5D={PageP}&tx_solr%5Bq%5D={KeywordP}",
}

# type c domains that do not have a working search function. Domains not yielding results in type A and B scraping will be added to fallback in main loop. 
SITEMAP_FALLBACK_DOMAINS = {
    5: "https://www.vci.de",
    15: "https://www.ruhr-uni-bochum.de",
    20: "https://www.basf.com",
    22: "https://www.eutop.com",
    26: "https://www.kalk.de",
    29: "https://www.cm-allianz.de",
    30: "https://carbon-management-initiative.de",
    32: "https://dvne.org",
    34: "https://gas-h2.de",
    35: "https://www.papierindustrie.de",
    36: "https://www.dvgw.de",
    37: "https://www.dr-martin-eckert.de",
    38: "https://www.thyssenkrupp-uhde.com",
    39: "https://www.thyssenkrupp-polysius.com",
    41: "https://www.equinor.de",
    45: "https://www.faktor3.de",
    48: "https://www.handundfusser.de",
    50: "https://germanzero.de",
    51: "https://www.h2ercules.com",
    52: "https://www.portofrotterdam.com",
    55: "https://www.ihk-nord.de",
    59: "https://www.linde-gas.de",
    64: "https://www.rostock-port.de",
    68: "https://www.thepartners.io",
    69: "https://tes-h2.com",
    75: "https://www.vais.de",
    78: "https://www.vdz-online.de",
    82: "https://www.vbcoll.de",
    85: "https://wirtschaftsrat.de",
    87: "https://zds-seehaefen.de",
    88: "https://www.bayernets.de",
    #added after testing result pages html lengths (same lengths for all kinds of keywords: non-working searches) (See manual_check_cells.ipynb)
    17: "https://www.dstgb.de",
    53: "https://www.helmholtz-klima.de",
    76: "https://www.vdma.eu",
    65: "https://www.rwe.com",
    #added after checking returned links overlap (100% overlap regardless of keyword: non-working searches) (See manual_check_cells.ipynb)
    44: "https://www.evonik.com",
    21: "https://www.bde.de",
    80: "https://vik.de",
    67: "https://www.swm.de",
    #added after checking returned links overlap (100% overlap or always 0 results: non-working searches) (See manual_check_cells.ipynb)
    43: "https://www.everllence.com",
    46: "https://www.fluxys.com",
    54: "https://www.holcim.de",
    58: "https://www.lhoist.com",
    61: "https://www.gasunie.nl",
    70: "https://www.tuev-nord.de",
    71: "https://www.tuv.com",
    73: "https://www.uniper.energy",
    84: "https://www.bernd-westphal.de",
}


In [6]:
# --- TEST: Brave Search HTML as Type C candidate (separate from main loop) ---

import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import quote, urljoin
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BRAVE_TEST_DOMAINS = [
    "www.basf.com",
    "www.vci.de",
    "www.thyssenkrupp-uhde.com"
]

BRAVE_TEST_KEYWORD = "CCS"

BRAVE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0 Safari/537.36",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def build_brave_search_url(domain, keyword):
    query = f"site:{domain} {keyword}"
    return f"https://search.brave.com/search?q={quote(query, safe='')}"

def parse_brave_results(soup):
    links = []

    # first targeted selectors
    selectors = [
        "a[data-test-id='result-title']",
        "a.snippet-title",
        "div.snippet a",
        "a[href]"
    ]

    for selector in selectors:
        for a in soup.select(selector):
            href = a.get("href")
            if not href:
                continue
            if href.startswith("#"):
                continue
            if href.startswith("javascript:"):
                continue
            if href.startswith("mailto:"):
                continue
            if href.startswith("tel:"):
                continue
            if not href.startswith("http"):
                continue
            links.append(href)

        if links:
            break

    # dedup
    return list(dict.fromkeys(links))

def detect_brave_block_or_challenge(response, soup):
    text_lower = response.text.lower()
    title_tag = soup.find("title")
    title_text = title_tag.get_text(strip=True) if title_tag else ""

    indicators = [
        "captcha",
        "verify you are human",
        "verification",
        "automated queries",
        "unusual traffic",
        "challenge",
        "access denied"
    ]

    hit = any(x in text_lower for x in indicators)

    return {
        "is_block_like": hit,
        "title": title_text,
        "status_code": response.status_code,
        "final_url": response.url,
        "html_length": len(response.text)
    }

brave_probe_rows = []

for domain in BRAVE_TEST_DOMAINS:
    search_url = build_brave_search_url(domain, BRAVE_TEST_KEYWORD)

    print(f"\n=== BRAVE TEST | {domain} | KW='{BRAVE_TEST_KEYWORD}' ===")
    print("URL:", search_url)

    try:
        response = requests.get(
            search_url,
            headers=BRAVE_HEADERS,
            timeout=15,
            verify=False
        )
        soup = BeautifulSoup(response.text, "html.parser")

        diag = detect_brave_block_or_challenge(response, soup)
        links = parse_brave_results(soup)

        print("Status:", diag["status_code"])
        print("Final URL:", diag["final_url"])
        print("Title:", diag["title"])
        print("HTML length:", diag["html_length"])
        print("Block-like page detected:", diag["is_block_like"])
        print("Found links:", len(links))

        for link in links[:10]:
            print(" -", link)

        brave_probe_rows.append({
            "domain": domain,
            "keyword": BRAVE_TEST_KEYWORD,
            "search_url": search_url,
            "status_code": diag["status_code"],
            "final_url": diag["final_url"],
            "title": diag["title"],
            "html_length": diag["html_length"],
            "block_like": diag["is_block_like"],
            "n_links": len(links),
            "links": links
        })

    except Exception as e:
        print("ERROR:", repr(e))
        brave_probe_rows.append({
            "domain": domain,
            "keyword": BRAVE_TEST_KEYWORD,
            "search_url": search_url,
            "status_code": None,
            "final_url": None,
            "title": None,
            "html_length": None,
            "block_like": None,
            "n_links": None,
            "links": [],
            "error": repr(e)
        })

    time.sleep(2.0)

df_brave_probe = pd.DataFrame(brave_probe_rows)
df_brave_probe


=== BRAVE TEST | www.basf.com | KW='CCS' ===
URL: https://search.brave.com/search?q=site%3Awww.basf.com%20CCS
Status: 200
Final URL: https://search.brave.com/search?q=site%3Awww.basf.com%20CCS
Title: site:www.basf.com CCS - Brave Search
HTML length: 197716
Block-like page detected: True
Found links: 20
 - https://energy-resources.basf.com/dam/jcr:25f6d0b3-8e10-44ae-89b0-ddfc49b9d87c/basf/energy-resources/gas-treatment/documents/State-of-the-Art---CCS-Technologies-2025.pdf
 - https://download.basf.com/p1/8a8081d09c704ed5019c74f375a00001/en/Sorbead%C2%AE_Air_for_CCS
 - https://www.basf.com/global/en/media/news-releases/2025/05/p-25-096
 - https://www.basf.com/global/de/media/news-releases/2023/02/p-23-143
 - https://chemical-catalysts-and-adsorbents.basf.com/dam/jcr:3e67edff-2bc5-47d0-88d2-dcbc5f453d9b/basf/chemical-catalysts-and-adsorbents/Sorbead-for-Natural-Gas/CO2-Drying-for-CSS-applications--DecarbonizationTechnology--202210.pdf
 - https://www.basf.com/cn/en/media/news-releases/cn/

,domain,keyword,search_url,status_code,final_url,title,html_length,block_like,n_links,links
0,www.basf.com,CCS,https://search.brave.com/search?q=site%3Awww.b...,200,https://search.brave.com/search?q=site%3Awww.b...,site:www.basf.com CCS - Brave Search,197716,True,20,[https://energy-resources.basf.com/dam/jcr:25f...
1,www.vci.de,CCS,https://search.brave.com/search?q=site%3Awww.v...,200,https://search.brave.com/search?q=site%3Awww.v...,site:www.vci.de CCS - Brave Search,198646,True,20,[https://www.vci.de/themen/energie-klima/klima...
2,www.thyssenkrupp-uhde.com,CCS,https://search.brave.com/search?q=site%3Awww.t...,200,https://search.brave.com/search?q=site%3Awww.t...,site:www.thyssenkrupp-uhde.com CCS - Brave Search,191148,True,23,[https://www.thyssenkrupp-uhde.com/en/products...


In [7]:
# --- TEST: Brave HTML stress test with anti-block safeguards ---

import requests
import time
import random
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---------- CONFIG ----------
BRAVE_STRESS_DOMAINS = [
    "www.basf.com",
    "www.vci.de",
    "www.thyssenkrupp-uhde.com",
    "www.kalk.de",
    "www.dvgw.de",
    "www.bayernets.de",
    "www.equinor.de",
    "www.linde-gas.de",
    "www.vdz-online.de",
    "www.rostock-port.de"
]

BRAVE_STRESS_KEYWORDS = [
    "CCS",
    "CCU",
    "carbon management"
]

BRAVE_MAX_RESULTS_PER_QUERY = 20
BRAVE_SLEEP_MIN = 4.0
BRAVE_SLEEP_MAX = 8.0
BRAVE_STOP_AFTER_N_BLOCKS = 2
BRAVE_TIMEOUT = 20

BRAVE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0 Safari/537.36",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# ---------- HELPERS ----------
def build_brave_search_url(domain, keyword):
    query = f"site:{domain} {keyword}"
    return f"https://search.brave.com/search?q={quote(query, safe='')}"

def parse_brave_results(soup):
    links = []

    selectors = [
        "a[data-test-id='result-title']",
        "a.snippet-title",
        "div.snippet a",
        "a[href]"
    ]

    for selector in selectors:
        for a in soup.select(selector):
            href = a.get("href")
            if not href:
                continue
            if href.startswith("#"):
                continue
            if href.startswith("javascript:"):
                continue
            if href.startswith("mailto:"):
                continue
            if href.startswith("tel:"):
                continue
            if not href.startswith("http"):
                continue
            links.append(href)

        if links:
            break

    return list(dict.fromkeys(links))[:BRAVE_MAX_RESULTS_PER_QUERY]

def detect_brave_block_or_challenge(response, soup):
    text_lower = response.text.lower()
    title_tag = soup.find("title")
    title_text = title_tag.get_text(strip=True) if title_tag else ""

    strong_indicators = [
        "captcha",
        "verify you are human",
        "automated queries",
        "unusual traffic",
        "access denied",
    ]

    is_block_like = (
        response.status_code in [403, 429, 503]
        or any(x in text_lower for x in strong_indicators)
    )

    return {
        "is_block_like": is_block_like,
        "title": title_text,
        "status_code": response.status_code,
        "final_url": response.url,
        "html_length": len(response.text)
    }

# ---------- RUN ----------
session = requests.Session()
session.headers.update(BRAVE_HEADERS)

stress_rows = []
block_counter = 0
request_counter = 0

for domain in BRAVE_STRESS_DOMAINS:
    for keyword in BRAVE_STRESS_KEYWORDS:

        search_url = build_brave_search_url(domain, keyword)
        request_counter += 1

        print(f"\n[{request_counter}] BRAVE STRESS | {domain} | KW='{keyword}'")
        print("URL:", search_url)

        try:
            response = session.get(
                search_url,
                timeout=BRAVE_TIMEOUT,
                verify=False
            )

            soup = BeautifulSoup(response.text, "html.parser")
            diag = detect_brave_block_or_challenge(response, soup)
            links = parse_brave_results(soup)

            print("Status:", diag["status_code"])
            print("Title:", diag["title"])
            print("HTML length:", diag["html_length"])
            print("Block-like page detected:", diag["is_block_like"])
            print("Found links:", len(links))

            if links:
                for link in links[:5]:
                    print(" -", link)

            if diag["is_block_like"]:
                block_counter += 1
                print(f"⚠️ Block counter: {block_counter}")

            stress_rows.append({
                "domain": domain,
                "keyword": keyword,
                "search_url": search_url,
                "status_code": diag["status_code"],
                "final_url": diag["final_url"],
                "title": diag["title"],
                "html_length": diag["html_length"],
                "block_like": diag["is_block_like"],
                "n_links": len(links),
                "links": links
            })

            if block_counter >= BRAVE_STOP_AFTER_N_BLOCKS:
                print("\n🛑 Early stop: too many block-like responses")
                break

            sleep_time = random.uniform(BRAVE_SLEEP_MIN, BRAVE_SLEEP_MAX)
            print(f"Sleep: {sleep_time:.2f}s")
            time.sleep(sleep_time)

        except Exception as e:
            print("ERROR:", repr(e))

            stress_rows.append({
                "domain": domain,
                "keyword": keyword,
                "search_url": search_url,
                "status_code": None,
                "final_url": None,
                "title": None,
                "html_length": None,
                "block_like": None,
                "n_links": None,
                "links": [],
                "error": repr(e)
            })

            sleep_time = random.uniform(BRAVE_SLEEP_MIN, BRAVE_SLEEP_MAX)
            print(f"Sleep after error: {sleep_time:.2f}s")
            time.sleep(sleep_time)

    if block_counter >= BRAVE_STOP_AFTER_N_BLOCKS:
        break

df_brave_stress = pd.DataFrame(stress_rows)
df_brave_stress


[1] BRAVE STRESS | www.basf.com | KW='CCS'
URL: https://search.brave.com/search?q=site%3Awww.basf.com%20CCS
Status: 429
Title: Brave Search
HTML length: 76133
Block-like page detected: True
Found links: 1
 - https://tb-manual.torproject.org/security-settings/#safest
⚠️ Block counter: 1
Sleep: 6.44s

[2] BRAVE STRESS | www.basf.com | KW='CCU'
URL: https://search.brave.com/search?q=site%3Awww.basf.com%20CCU
Status: 429
Title: Brave Search
HTML length: 76133
Block-like page detected: True
Found links: 1
 - https://tb-manual.torproject.org/security-settings/#safest
⚠️ Block counter: 2

🛑 Early stop: too many block-like responses


,domain,keyword,search_url,status_code,final_url,title,html_length,block_like,n_links,links
0,www.basf.com,CCS,https://search.brave.com/search?q=site%3Awww.b...,429,https://search.brave.com/search?q=site%3Awww.b...,Brave Search,76133,True,1,[https://tb-manual.torproject.org/security-set...
1,www.basf.com,CCU,https://search.brave.com/search?q=site%3Awww.b...,429,https://search.brave.com/search?q=site%3Awww.b...,Brave Search,76133,True,1,[https://tb-manual.torproject.org/security-set...


In [8]:
# Test Sitemap Method for Type C URLs

import requests
import re
import xml.etree.ElementTree as ET
import urllib3

# suppress warnings for verify=false
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

SITEMAP_MAX_URLS_PER_DOMAIN = 10000
SITEMAP_TIMEOUT = 10


# ---------------------------
# Sitemap Parsing
# ---------------------------

def parse_sitemap_xml(xml_text):
    root = ET.fromstring(xml_text)

    urls = []
    sitemaps = []

    tag = root.tag.lower()

    if "sitemapindex" in tag:
        for elem in root.iter():
            if elem.tag.lower().endswith("loc") and elem.text:
                sitemaps.append(elem.text.strip())

    elif "urlset" in tag:
        for elem in root.iter():
            if elem.tag.lower().endswith("loc") and elem.text:
                loc = elem.text.strip()

                # fix for encoded URLs
                loc = loc.replace("https%3A//", "https://").replace("http%3A//", "http://")

                # for doubly-composite URLS: cut before second http(s)
                second_https = loc.find("https://", 8)
                second_http = loc.find("http://", 7)

                candidates = [pos for pos in [second_https, second_http] if pos != -1]
                if candidates:
                    loc = loc[min(candidates):]

                urls.append(loc)

    return sitemaps, urls


# ---------------------------
# Sitemap Collector (with SSL-Retry)
# ---------------------------

def get_sitemap_candidates(base_url):
    base_url = base_url.rstrip("/")

    return [
        f"{base_url}/sitemap.xml",
        f"{base_url}/sitemap_index.xml",
        f"{base_url}/sitemap-index.xml"
    ]


def collect_urls_from_sitemap(start_url, max_urls=SITEMAP_MAX_URLS_PER_DOMAIN):
    to_visit = [start_url]
    collected_urls = []

    while to_visit and len(collected_urls) < max_urls:
        sitemap_url = to_visit.pop(0)

        ssl_retry_used = False

        try:
            resp = requests.get(
                sitemap_url,
                timeout=SITEMAP_TIMEOUT,
                verify=True
            )
            resp.raise_for_status()

        except requests.exceptions.SSLError:
            try:
                resp = requests.get(
                    sitemap_url,
                    timeout=SITEMAP_TIMEOUT,
                    verify=False
                )
                resp.raise_for_status()
                ssl_retry_used = True
                print(f"   ssl retry used: {sitemap_url}")
            except Exception as e:
                print(f"   sitemap read failed: {sitemap_url} | {repr(e)}")
                continue

        except Exception as e:
            print(f"   sitemap read failed: {sitemap_url} | {repr(e)}")
            continue

        try:
            new_sitemaps, page_urls = parse_sitemap_xml(resp.text)
        except Exception as e:
            print(f"   xml parse failed: {sitemap_url} | {repr(e)}")
            continue

        to_visit.extend(new_sitemaps)

        for u in page_urls:
            collected_urls.append(u)

            if len(collected_urls) >= max_urls:
                break

    return collected_urls


# ---------------------------
# Keyword Matching
# ---------------------------

def url_keyword_match(url, keyword):
    url_l = url.lower()
    keyword_l = keyword.lower().strip()

    patterns = [
        f"/{keyword_l}",
        f"-{keyword_l}",
        f"_{keyword_l}",
        f"{keyword_l}/",
    ]

    if any(p in url_l for p in patterns):
        return True

    return keyword_l in url_l


def score_url_only(url, keyword):
    score = 0
    url_l = url.lower()

    if url_keyword_match(url, keyword):
        score += 10

    if "/de/" in url_l or url_l.endswith("/de"):
        score += 1

    if any(x in url_l for x in ["/news", "/presse", "/media", "/themen", "/topics", "/insights"]):
        score += 1

    return score


def find_keyword_hits_in_urls(urls, keyword, max_hits=20):
    hits = []

    for url in urls:
        if url_keyword_match(url, keyword):
            hits.append({
                "found_url": url,
                "keyword": keyword,
                "score": score_url_only(url, keyword)
            })

    hits = sorted(hits, key=lambda x: (-x["score"], x["found_url"]))
    return hits[:max_hits]


# ---------------------------
# TEST RUN
# ---------------------------

test_domains = [
    "https://www.basf.com",
    "https://www.vci.de",
    "https://www.thyssenkrupp-uhde.com"
]

test_keywords = ["CCS", "CCU", "carbon management"]

for domain in test_domains:
    print(f"\n=== SITEMAP TEST | {domain} ===")

    candidates = get_sitemap_candidates(domain)
    print("Candidate sitemaps:")
    for c in candidates:
        print(" -", c)

    all_page_urls = []

    for c in candidates:
        urls = collect_urls_from_sitemap(c)
        if urls:
            all_page_urls = urls
            break

    print(f"Collected page URLs: {len(all_page_urls)}")

    for keyword in test_keywords:
        print(f"   Keyword: {keyword}")

        hits = find_keyword_hits_in_urls(all_page_urls, keyword, max_hits=20)

        print(f"   URL Hits: {len(hits)}")
        for hit in hits[:5]:
            print("    -", hit["found_url"], f"(score={hit['score']})")


=== SITEMAP TEST | https://www.basf.com ===
Candidate sitemaps:
 - https://www.basf.com/sitemap.xml
 - https://www.basf.com/sitemap_index.xml
 - https://www.basf.com/sitemap-index.xml
   sitemap read failed: https://www.basf.com/dam/sitemaps/basf/www/mx/es.xml | HTTPError('404 Client Error: Not Found for url: https://www.basf.com/dam/sitemaps/basf/www/mx/es.xml')
Collected page URLs: 10000
   Keyword: CCS
   URL Hits: 4
    - https://www.basf.com/cn/zh/media/news-releases/cn/2018/05/CCS (score=11)
    - https://www.basf.com/cn/zh/media/news-releases/cn/2021/04/CCS_BASF_2021 (score=11)
    - https://www.basf.com/cn/zh/media/news-releases/cn/2023/06/CCS_BASF_Youth_Innovation_Prize (score=11)
    - https://www.basf.com/cn/zh/media/news-releases/global/2021/11/Air_Liquide_and_BASF_welcome_support_from_European_Innovation_Fund_for_joint_CCS_project (score=11)
   Keyword: CCU
   URL Hits: 20
    - https://www.basf.com/cn/zh/media/GC-report/GC-report-2022/basf-in-greater-china/EHS/occupation

In [9]:
#9.1 Manual check for static result pages 
test_cases = {
    17: "https://www.dstgb.de/?q={}",
    44: "https://www.evonik.com/de/search.html?q={}&languages=German%3BEnglish",
    53: "https://www.helmholtz-klima.de/#o=search&q={}",
    76: "https://www.vdma.eu/de/suche#search-page-tabbar-articles-id?searchKeyword={}&sorts=RELEVANCE",
    65: "https://www.rwe.com/suche/?q={}",
    21: "https://www.bde.de/search/?q={}"
}

test_keywords = ["CCS", "banana", "asldkfjalskdf"]  # random by choice to test for genuine results

import requests

for source_id, url_template in test_cases.items():
    print(f"\n=== TEST {source_id} ===")

    for kw in test_keywords:
        url = url_template.format(kw)

        try:
            r = requests.get(url, timeout=10, verify=False)
            print(f"{kw:15} → status={r.status_code} | length={len(r.text)}")
        except Exception as e:
            print(f"{kw:15} → ERROR: {e}")


=== TEST 17 ===
CCS             → status=200 | length=374792
banana          → status=200 | length=374790
asldkfjalskdf   → status=200 | length=374801

=== TEST 44 ===
CCS             → status=200 | length=371539
banana          → status=200 | length=371545
asldkfjalskdf   → status=200 | length=371559

=== TEST 53 ===
CCS             → status=200 | length=60175
banana          → status=200 | length=60175
asldkfjalskdf   → status=200 | length=60175

=== TEST 76 ===
CCS             → status=200 | length=200335
banana          → status=200 | length=200335
asldkfjalskdf   → status=200 | length=200335

=== TEST 65 ===
CCS             → status=403 | length=585
banana          → status=403 | length=585
asldkfjalskdf   → status=403 | length=585

=== TEST 21 ===
CCS             → status=200 | length=33481
banana          → status=200 | length=24244
asldkfjalskdf   → status=200 | length=24272


In [10]:
#9.1  -------- MANUAL CHECK TYPE A: ARE THE SAME LINKS RETURNED? --------

MANUAL_TEST_CASES = {
    44: {
        "name": "Evonik",
        "url_template": "https://www.evonik.com/de/search.html?q={KeywordP}&languages=German%3BEnglish"
    },
    21: {
        "name": "BDE",
        "url_template": "https://www.bde.de/search/?q={KeywordP}"
    },
    67: {
        "name": "SWM",
        "url_template": "https://www.swm.de/suche?suchbegriff={KeywordP}"
    },
    80: {
        "name": "VIK",
        "url_template": "https://vik.de/search?search={KeywordP}"
    }
}

MANUAL_TEST_KEYWORDS = ["CCS", "Carbon Management", "banana", "asldkfjalskdf"]
TOP_N_COMPARE = 15

manual_compare_results = []

for source_id, meta in MANUAL_TEST_CASES.items():
    print(f"\n{'='*80}")
    print(f"[{source_id}] {meta['name']}")
    print(f"{'='*80}")

    keyword_to_links = {}

    for keyword in MANUAL_TEST_KEYWORDS:
        keyword_enc = quote(keyword, safe="")
        full_url = meta["url_template"].format(KeywordP=keyword_enc)

        ssl_retry_used = False

        try:
            kwargs = get_request_kwargs(full_url)

            try:
                response = requests.get(full_url, timeout=10, **kwargs)
            except requests.exceptions.SSLError as e:
                print(f"⚠️ SSL-Fehler -> Retry ohne Verifikation: {full_url}")
                print(f"   → {type(e).__name__}: {e}")

                response = requests.get(
                    full_url,
                    timeout=10,
                    verify=False,
                    headers=DEFAULT_HEADERS
                )

                ssl_retry_domains.add(urlparse(full_url).netloc)
                ssl_retry_urls.append(full_url)
                ssl_retry_used = True

            soup = BeautifulSoup(response.text, "html.parser")
            links = parse_normal_page(soup, full_url)
            links_top = links[:TOP_N_COMPARE]

            keyword_to_links[keyword] = links_top

            print(f"\nKW='{keyword}' | n_links={len(links)} | top_{TOP_N_COMPARE} shown={len(links_top)} | ssl_retry_used={ssl_retry_used}")
            for i, link in enumerate(links_top[:10], start=1):
                print(f"  {i:02d}. {link}")

        except Exception as e:
            keyword_to_links[keyword] = []
            print(f"\nKW='{keyword}' -> ERROR: {type(e).__name__}: {e}")

    # pairwise overlap
    kws = list(keyword_to_links.keys())
    for i in range(len(kws)):
        for j in range(i + 1, len(kws)):
            kw1 = kws[i]
            kw2 = kws[j]

            set1 = set(keyword_to_links[kw1])
            set2 = set(keyword_to_links[kw2])

            if len(set1) == 0 and len(set2) == 0:
                overlap_ratio = None
                overlap_count = 0
            else:
                overlap_count = len(set1 & set2)
                overlap_ratio = overlap_count / max(len(set1), len(set2)) if max(len(set1), len(set2)) > 0 else None

            manual_compare_results.append({
                "source_id": source_id,
                "name": meta["name"],
                "kw1": kw1,
                "kw2": kw2,
                "top_n": TOP_N_COMPARE,
                "overlap_count": overlap_count,
                "overlap_ratio": overlap_ratio
            })

print(f"\n{'='*80}")
print("PAIRWISE OVERLAP SUMMARY")
print(f"{'='*80}")

df_manual_compare = pd.DataFrame(manual_compare_results)
display(df_manual_compare.sort_values(["source_id", "overlap_ratio"], ascending=[True, False]))


[44] Evonik

KW='CCS' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='Carbon Management' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='banana' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='asldkfjalskdf' -> ERROR: NameError: name 'get_request_kwargs' is not defined

[21] BDE

KW='CCS' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='Carbon Management' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='banana' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='asldkfjalskdf' -> ERROR: NameError: name 'get_request_kwargs' is not defined

[67] SWM

KW='CCS' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='Carbon Management' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='banana' -> ERROR: NameError: name 'get_request_kwargs' is not defined

KW='asldkfjalskdf' -> ERROR: NameError: name 'get_request_kwargs' is not defined

[80] VIK

KW='CCS' ->

,source_id,name,kw1,kw2,top_n,overlap_count,overlap_ratio
6,21,BDE,CCS,Carbon Management,15,0,None
7,21,BDE,CCS,banana,15,0,None
8,21,BDE,CCS,asldkfjalskdf,15,0,None
9,21,BDE,Carbon Management,banana,15,0,None
10,21,BDE,Carbon Management,asldkfjalskdf,15,0,None
11,21,BDE,banana,asldkfjalskdf,15,0,None
0,44,Evonik,CCS,Carbon Management,15,0,None
1,44,Evonik,CCS,banana,15,0,None
2,44,Evonik,CCS,asldkfjalskdf,15,0,None
3,44,Evonik,Carbon Management,banana,15,0,None


In [11]:
# MANUAL CHECK TYPE B
# checks if different search terms actually yield different hits

from itertools import combinations
from urllib.parse import quote
from collections import OrderedDict

TEST_KEYWORDS_B = ["CCS", "Carbon Management", "banana", "asldkfjalskdf"]

TEST_SOURCE_IDS_B = [2, 3, 6, 7, 9, 16, 18, 23, 24, 28, 31, 43, 57, 58, 66, 70, 71, 84]

TOP_N_COMPARE = 15
TEST_PAGE_B = 1


def unique_keep_order(seq):
    return list(OrderedDict.fromkeys(seq))


def run_type_b_manual_check(source_id, url_template, test_keywords, test_page=1, top_n=15):
    print("=" * 80)
    print(f"[{source_id}] {url_template}")
    print("=" * 80)

    results = {}

    for kw in test_keywords:
        keyword_enc = quote(kw, safe="")
        full_url = url_template.format(KeywordP=keyword_enc, PageP=test_page)

        ssl_retry_used = False

        try:
            kwargs = get_request_kwargs(full_url)

            try:
                response = requests.get(full_url, timeout=10, **kwargs)

            except requests.exceptions.SSLError as e:
                print(f"⚠️ SSL-Fehler -> Retry ohne Verifikation: {full_url}")
                print(f"   → {type(e).__name__}: {e}")

                response = requests.get(
                    full_url,
                    timeout=10,
                    verify=False,
                    headers=DEFAULT_HEADERS
                )
                ssl_retry_used = True

            soup = BeautifulSoup(response.text, "html.parser")
            links = parse_normal_page(soup, full_url)
            links = unique_keep_order(links)

            results[kw] = links

            print(f"\nKW='{kw}' | n_links={len(links)} | top_{top_n} shown={min(top_n, len(links))} | ssl_retry_used={ssl_retry_used}")
            for i, link in enumerate(links[:top_n], start=1):
                print(f"  {i:02d}. {link}")

        except Exception as e:
            results[kw] = []
            print(f"\nKW='{kw}' | ERROR")
            print(f"   → {type(e).__name__}: {e}")

    print("\n" + "=" * 80)
    print("PAIRWISE OVERLAP SUMMARY")
    print("=" * 80)

    overlap_rows = []

    for kw1, kw2 in combinations(test_keywords, 2):
        top1 = set(results[kw1][:top_n])
        top2 = set(results[kw2][:top_n])

        overlap = len(top1.intersection(top2))
        denom = min(len(top1), len(top2))
        ratio = overlap / denom if denom > 0 else None

        overlap_rows.append({
            "source_id": source_id,
            "kw1": kw1,
            "kw2": kw2,
            "top_n": top_n,
            "overlap_count": overlap,
            "overlap_ratio": ratio
        })

    overlap_df = pd.DataFrame(overlap_rows)
    print(overlap_df)

    return results, overlap_df


all_overlap_results_b = []

for source_id in TEST_SOURCE_IDS_B:
    if source_id not in urls:
        print(f"[{source_id}] not found in urls\n")
        continue

    results_b, overlap_df_b = run_type_b_manual_check(
        source_id=source_id,
        url_template=urls[source_id],
        test_keywords=TEST_KEYWORDS_B,
        test_page=TEST_PAGE_B,
        top_n=TOP_N_COMPARE
    )

    all_overlap_results_b.append(overlap_df_b)

if all_overlap_results_b:
    final_overlap_df_b = pd.concat(all_overlap_results_b, ignore_index=True)

    print("\n" + "=" * 80)
    print("ALL TYPE B OVERLAP RESULTS SORTED")
    print("=" * 80)

    display(final_overlap_df_b.sort_values(
        by=["overlap_ratio", "overlap_count", "source_id"],
        ascending=[False, False, True]
    ))

[2] not found in urls

[3] not found in urls

[6] not found in urls

[7] not found in urls

[9] not found in urls

[16] not found in urls

[18] not found in urls

[23] not found in urls

[24] not found in urls

[28] not found in urls

[31] not found in urls

[43] not found in urls

[57] not found in urls

[58] not found in urls

[66] not found in urls

[70] not found in urls

[71] not found in urls

[84] not found in urls



In [12]:
# MANUAL CHECK TYPE B
# checks if different search terms actually yield different hits

from itertools import combinations
from urllib.parse import quote
from collections import OrderedDict

TEST_KEYWORDS_B = ["CCS", "Carbon Management", "banana", "asldkfjalskdf"]


TEST_SOURCE_IDS_B = [2, 3, 6, 7, 9, 16, 18, 23, 24, 28, 31, 43, 57, 58, 66, 70, 71, 84]

TOP_N_COMPARE = 500
TEST_PAGE_B = 1


def unique_keep_order(seq):
    return list(OrderedDict.fromkeys(seq))


def run_type_b_manual_check(source_id, url_template, test_keywords, test_page=1, top_n=15):
    print("=" * 80)
    print(f"[{source_id}] {url_template}")
    print("=" * 80)

    results = {}

    for kw in test_keywords:
        keyword_enc = quote(kw, safe="")
        full_url = url_template.format(KeywordP=keyword_enc, PageP=test_page)

        ssl_retry_used = False

        try:
            kwargs = get_request_kwargs(full_url)

            try:
                response = requests.get(full_url, timeout=10, **kwargs)

            except requests.exceptions.SSLError as e:
                print(f"⚠️ SSL-Fehler -> Retry ohne Verifikation: {full_url}")
                print(f"   → {type(e).__name__}: {e}")

                response = requests.get(
                    full_url,
                    timeout=10,
                    verify=False,
                    headers=DEFAULT_HEADERS
                )
                ssl_retry_used = True

            soup = BeautifulSoup(response.text, "html.parser")
            links = parse_normal_page(soup, full_url)
            links = unique_keep_order(links)

            results[kw] = links

            print(f"\nKW='{kw}' | n_links={len(links)} | top_{top_n} shown={min(top_n, len(links))} | ssl_retry_used={ssl_retry_used}")
            for i, link in enumerate(links[:top_n], start=1):
                print(f"  {i:02d}. {link}")

        except Exception as e:
            results[kw] = []
            print(f"\nKW='{kw}' | ERROR")
            print(f"   → {type(e).__name__}: {e}")

    print("\n" + "=" * 80)
    print("PAIRWISE OVERLAP SUMMARY")
    print("=" * 80)

    overlap_rows = []

    for kw1, kw2 in combinations(test_keywords, 2):
        top1 = set(results[kw1][:top_n])
        top2 = set(results[kw2][:top_n])

        overlap = len(top1.intersection(top2))
        denom = min(len(top1), len(top2))
        ratio = overlap / denom if denom > 0 else None

        overlap_rows.append({
            "source_id": source_id,
            "kw1": kw1,
            "kw2": kw2,
            "top_n": top_n,
            "overlap_count": overlap,
            "overlap_ratio": ratio
        })

    overlap_df = pd.DataFrame(overlap_rows)
    print(overlap_df)

    return results, overlap_df


all_overlap_results_b = []

for source_id in TEST_SOURCE_IDS_B:
    if source_id not in urls:
        print(f"[{source_id}] not found in urls\n")
        continue

    results_b, overlap_df_b = run_type_b_manual_check(
        source_id=source_id,
        url_template=urls[source_id],
        test_keywords=TEST_KEYWORDS_B,
        test_page=TEST_PAGE_B,
        top_n=TOP_N_COMPARE
    )

    all_overlap_results_b.append(overlap_df_b)

if all_overlap_results_b:
    final_overlap_df_b = pd.concat(all_overlap_results_b, ignore_index=True)

    print("\n" + "=" * 80)
    print("ALL TYPE B OVERLAP RESULTS SORTED")
    print("=" * 80)

    display(final_overlap_df_b.sort_values(
        by=["overlap_ratio", "overlap_count", "source_id"],
        ascending=[False, False, True]
    ))

[2] not found in urls

[3] not found in urls

[6] not found in urls

[7] not found in urls

[9] not found in urls

[16] not found in urls

[18] not found in urls

[23] not found in urls

[24] not found in urls

[28] not found in urls

[31] not found in urls

[43] not found in urls

[57] not found in urls

[58] not found in urls

[66] not found in urls

[70] not found in urls

[71] not found in urls

[84] not found in urls



In [13]:
# retrieve domains from non-working searches

print("    # added after Type B manual checks (verified domains)")
for sid in [43, 46, 54, 58, 61, 70, 71, 73, 84]:
    if sid not in urls:
        print(f"    # {sid} not found in urls")
        continue

    base = urls[sid].split("?", 1)[0].split("#", 1)[0]
    parsed = urlparse(base)
    base_domain = f"{parsed.scheme}://{parsed.netloc}"

    print(f'    {sid}: "{base_domain}",')

    # added after Type B manual checks (verified domains)
    # 43 not found in urls
    # 46 not found in urls
    # 54 not found in urls
    # 58 not found in urls
    # 61 not found in urls
    # 70 not found in urls
    # 71 not found in urls
    # 73 not found in urls
    # 84 not found in urls
